In [ ]:
%load_ext autoreload
%autoreload 2
import jax.numpy as jnp
import jax_sn

import jax_sbgeom as jsb
import jax
import matplotlib.pyplot as plt
import numpy as onp
import equinox as eqx
jax.config.update("jax_enable_x64", True)
from jax_sbgeom.flux_surfaces import FluxSurfaceNormalExtendedNoPhi, ToroidalExtent, FluxSurface
from jax_sn.tools.memory_analysis import _filter_jit_f_to_memory
vmec_file = ["/home/tbogaarts/stellarator_paper/base_data/vmecs/helias5_vmec.nc4"][0]
fs_base = FluxSurfaceNormalExtendedNoPhi.from_hdf5(vmec_file)
from functools import partial
import pyvista as pv
from jax_sbgeom.jax_utils import mesh_to_pyvista_mesh
import lineax as lx
from jax_sn.operators.multi_group_operator import MultiGroupLowerTriangularBlockSolver, MultiGroupOperatorScanCheckpoint
from jax_sn.solution_domain import SolutionDomainData
from typing import Tuple, Callable, Iterable
from dataclasses import dataclass
from breeding_blanket_setup import _get_bb_args, _setup_bb_mgop_symmetric, aux_data_dict, bb_data_dict


In [ ]:
from jax_sn.operators.multi_group_operator import MultiGroupLowerTriangularBlockSolver, MultiGroupOperator
from jax_sn.operators.multi_group_operator.multi_group_operator_imp import _solve_multigroup_operator_rhs_scan
from jax_sn.parametrisation.multigroup_parametrisation import MultigroupOperatorParametrisedObjective, MultigroupOperatorParametrisedVerticesNoToposort
from jax_sn.tools.parametrisation_tools import ParametrisationValues, ParametrisedModuleTheta
from jax_sn.operators.transport_operator import TransportOperatorVmapPallas, TransportOperatorMultipleAnglesVmap
import mcbb
from mcbb.linear_blanket import OpenMCLinearBlanketSimulation, OpenMCSymmetricLinearBlanketSimulation
import openmc
from functools import lru_cache

In [ ]:

def create_openmc_data(blanket : mcbb.linear_blanket.Blanket1D, groups_list : list, settings : mcbb.blanket_base.OpenMCSettingsBlanket, l_order, n_x_tally = 200) -> Tuple[dict, dict, dict]:
    simulation_symm = OpenMCSymmetricLinearBlanketSimulation(blanket, settings, n_x_tally, groups_list)
    mgxs_lib, tally_result_dict =simulation_symm.create_mgxs_library(onp.flip(groups_list), l_order)
    mgxs_lib_new = mcbb.openmc_base_functions.mgxs_lib_to_standard_formatting(mgxs_lib)
    new_bb_data_dict = {key : (mgxs_lib_new[key]['total'], mgxs_lib_new[key]['nu-scatter matrix']) for key in bb_data_dict().keys()}
    new_aux_data_dict = {key : {'nt' : mgxs_lib_new[key]['(n,Xt)']} for key in aux_data_dict().keys()}
    return new_bb_data_dict, new_aux_data_dict, tally_result_dict

@lru_cache(maxsize=  4)
def _get_hcpb_data_openmc(d_elem_regions : Iterable[float], l_order : int, n_samples_per_batch : int, groups_list : Iterable[float]) -> Tuple[dict, dict, dict]:
    # Returns bb_data_dict, aux_data_dict, tally_result_dict
    hcpb_blanket = mcbb.examples.hcpb()
    for i, layer in enumerate(hcpb_blanket.layers):
        if i == 0:
            layer.thickness = 0.5 * d_elem_regions[i]
        else:
            layer.thickness = d_elem_regions[i]
    settings = mcbb.blanket_base.OpenMCSettingsBlanket(True, "openmc_data/", 10, n_samples_per_batch)
    return create_openmc_data(hcpb_blanket, groups_list, settings, l_order)


In [ ]:
class BZShieldValues(ParametrisationValues):
    bzshield : jnp.ndarray

class BZShieldSettings(eqx.Module):
    resolution_layers : Tuple[int]
    base_layer_size   : jnp.ndarray
    
from breeding_blanket_setup import recreate_symmetric_vertices

@eqx.filter_jit
def create_bzshield_vertices(params : BZShieldValues, settings : BZShieldSettings):
    bzshield = params.bzshield
    resolution_layers = settings.resolution_layers
    base_layer_size   = settings.base_layer_size    
    size              =   base_layer_size.at[3].set(bzshield[0]).at[6].set(bzshield[1])    
    return recreate_symmetric_vertices(size, resolution_layers)[:, None]
    
class BZShieldParametrisation(ParametrisedModuleTheta):

    additional_settings : BZShieldSettings

    def __check_init__(self):
        assert isinstance(self.base_module, MultiGroupOperator), "Base module must be a MultiGroupOperator."
        assert isinstance(self.additional_settings, BZShieldSettings), "Additional settings must be of type BZShieldSettings."
    
    @eqx.filter_jit
    def to_module(self, theta : BZShieldValues):
        return MultigroupOperatorParametrisedVerticesNoToposort.reconstruct_operator(self.base_module, create_bzshield_vertices(theta, self.additional_settings))
     
def get_solution_domain(MGOP, element):
    return jax_sn.solution_domain.SolutionDomain(MGOP.transport_operator.transport_operator.solution_domain_data, element)

def fast_neutron_flux(solution_values, Ij, energy_set_boundaries, fast_eg = [14.2e6, 1e6]):
    last_element = solution_values.shape[-2] - 1    
    last_element_solvals = solution_values[..., 0, last_element, :] #[n_energy_groups, n_basis_per_element]
    overlap_matrix       = jax_sn.energy_set.compute_overlap_matrix(energy_set_boundaries, fast_eg)
    egs_fast             = overlap_matrix @ last_element_solvals  #[1, n_basis_per_element]
    return jnp.einsum("ij, j->", egs_fast, Ij)  / jnp.sqrt(4 * jnp.pi) * 1e2

def distance(theta : BZShieldValues, settings : BZShieldSettings):
    bzshield          = theta.bzshield    
    base_layer_size   = settings.base_layer_size    
    size              = base_layer_size.at[3].set(bzshield[0]).at[6].set(bzshield[1])    
    return jnp.sum(size[1:])

def tritium_breeding_ratio(solution_values, settings :BZShieldSettings, solution_domain_data : SolutionDomainData, tritium_xs):
    bz_elements = sum(settings.resolution_layers) + sum(settings.resolution_layers[1:3]) # sum all previous layers (including plasma) and then first two actual blanket layers        
    solution_values_bz = jax.lax.dynamic_slice(solution_values[:,0, :, : ], (0, bz_elements, 0 ), (solution_values.shape[0], settings.resolution_layers[3], solution_values.shape[-1])) #[n_energy_groups, n_elements_bz, n_basis_per_element]
    tritium_xs_bz  = tritium_xs['BreedingZone']['nt']
    jacobian_dets = jax.lax.dynamic_slice(solution_domain_data.precomputed_geometric_data.vol_jacobians_det, (bz_elements,), (settings.resolution_layers[3],))  #[n_elements_bz,]
    return jnp.einsum("g, geb, b, e -> ", tritium_xs_bz, solution_values_bz, solution_domain_data.fem_matrices.integration_coeff, jacobian_dets) / jnp.sqrt(4 * jnp.pi) / settings.base_layer_size[0] * 2.0 # / jnp.sqrt(4 * jnp.pi) to account for flux to scalar flux, / base_layer_size[0] to account for normalization over area, *2 for symmetry

class BZShieldMultiGroupObjective(MultigroupOperatorParametrisedObjective):
    
    aux_data : dict
    energy_set_boundaries : Tuple[float]

    
    @eqx.filter_jit
    def objective_and_constraints(self, theta : BZShieldValues):
        mgop_theta = self.theta_parametrised_module.to_module(theta)
        solution_values = self._to_solution(mgop_theta).value
        return (tritium_breeding_ratio(solution_values, self.theta_parametrised_module.additional_settings, mgop_theta.transport_operator.transport_operator.solution_domain_data, self.aux_data),
                fast_neutron_flux(solution_values, mgop_theta.transport_operator.solution_domain_data.fem_matrices.integration_coeff, self.energy_set_boundaries))
        return (self.distance(theta), 
                tritium_breeding_ratio(solution_values, self.theta_parametrised_module.additional_settings, mgop_theta.transport_operator.transport_operator.solution_domain_data, self.aux_data),
                fast_neutron_flux(solution_values, mgop_theta.transport_operator.solution_domain_data.fem_matrices.integration_coeff, self.energy_set_boundaries))

    def tritium_breeding_ratio(self, theta : BZShieldValues):
        mgop_theta = self.theta_parametrised_module.to_module(theta)
        solution_values = self._to_solution(mgop_theta).value
        return tritium_breeding_ratio(solution_values, self.theta_parametrised_module.additional_settings, mgop_theta.transport_operator.transport_operator.solution_domain_data, self.aux_data)
    
    def edge_fast_flux(self, theta : BZShieldValues):
        mgop_theta = self.theta_parametrised_module.to_module(theta)
        solution_values = self._to_solution(mgop_theta).value
        return fast_neutron_flux(solution_values, mgop_theta.transport_operator.solution_domain_data.fem_matrices.integration_coeff, self.energy_set_boundaries)
    
    def distance(self, theta : BZShieldValues):
        return distance(theta, self.theta_parametrised_module.additional_settings)

In [ ]:
def mgop_base(dim, degree, d_elem_regions, n_elem_regions, d_other_dim, n_other_dim, tn_order, bb_data_dict):
    MGOP, source_total, solution_domain = _setup_bb_mgop_symmetric(dim, degree, d_elem_regions, n_elem_regions, d_other_dim, n_other_dim, tn_order,  bb_data_dict)
    pallas_op       = TransportOperatorMultipleAnglesVmap(TransportOperatorVmapPallas(MGOP.transport_operator.transport_operator.solution_domain_data, MGOP.transport_operator.transport_operator.topological_sorted_angle, 0))
    MGOPcp          = MultiGroupOperatorScanCheckpoint(MGOP.moment_operator, MGOP.discrete_operator, pallas_op, MGOP.mass_operator, MGOP.gather_scatter_operator, MGOP.n_energy_groups, MGOP.transpose_bool)
    source_all_eg   = jnp.stack([source_total if i==0 else jnp.zeros_like(source_total) for i in range(MGOP.n_energy_groups)])
    solver          = MultiGroupLowerTriangularBlockSolver(lx.BiCGStab(1e-12,1e-12, max_steps= 100))
    return MGOPcp, source_all_eg, solver

def setup_multigroup_objective(dim, degree, d_elem_regions, n_elem_regions, d_other_dim, n_other_dim, tn_order, bb_data_dict, aux_data_dict, groups_list):
    MGOPcp, source_all_eg, solver = mgop_base(dim, degree, d_elem_regions, n_elem_regions, d_other_dim, n_other_dim, tn_order, bb_data_dict)
    additional_settings      = BZShieldSettings(tuple(n_elem_regions), jnp.array(d_elem_regions))
    operator_parametrisation = BZShieldParametrisation(MGOPcp, additional_settings)    
    return BZShieldMultiGroupObjective(operator_parametrisation, source_all_eg, solver , aux_data= aux_data_dict, energy_set_boundaries= tuple(groups_list))

def get_element(dim,degree):
    return jax_sn.solution_domain.BasixLagrangianSimplex(degree, dim)


In [ ]:
# BASE Parameters
dim = 1 
degree = 7
tn_order = 3
n_elem_regions = [5, 1, 1, 5, 5, 1, 5, 2]
d_other_dim = 10.0
n_other_dim = 1
finite_element = get_element(dim, degree)

# Variable parameters
groups_list = (14.1e6, 10e6, 3e6, 1e6, 1e4, 300, 10,0)
d_elem_regions_base = (0.2, 0.002, 0.025, 0.5, 0.385, 0.06, 0.2, 0.06)
new_bb_data_dict, new_aux_data_dict, tally_result_dict = _get_hcpb_data_openmc(d_elem_regions_base, 5, 1e4, groups_list)

values_init = BZShieldValues(jnp.array([d_elem_regions_base[3], d_elem_regions_base[6]]))

multigroup_objective = setup_multigroup_objective(dim, degree, d_elem_regions_base, n_elem_regions, d_other_dim, n_other_dim, tn_order, new_bb_data_dict, new_aux_data_dict, groups_list)

In [ ]:
from jax_sn.energy_set import compute_overlap_matrix
overlap_matrix = compute_overlap_matrix(groups_list , [14.2e6,1e6])

In [ ]:
import lineax as lx 

A_full = jnp.array([[3., 1., 0.],
                    [1., 2., 1.],
                    [0., 1., 1.]])


A_null = A_null = jnp.array([[1., 2., -1.],
                    [2., 4., -2.],
                    [3., 6., -3.]])  # rank 1
# Simple RHS
b      = jnp.array([1., 2., 3.])
Aop    = lx.MatrixLinearOperator(A_full)
solver = lx.BiCGStab(1e-12,1e-12, max_steps= 500)

def objective_1(b, solver):
    x = lx.linear_solve(Aop, b, solver)
    return jnp.sum(x.value)

def objective(b, solver):
    x = lx.linear_solve(Aop, b, solver)
    return jnp.sum(b), jnp.sum(x.value)
grad_objective_1 = jax.jacrev(objective_1, argnums=0)
grad_objective = jax.jacrev(lambda b: objective(b, solver))

try:
    print(grad_objective_1(b, solver))
    print("Single function return works")
except:
    print("Single function return does not work")

try:
    print(grad_objective(b))
    print("Multiple function return works")
except Exception as e:
    print(e)
    print("Multiple function return does not work")


In [ ]:
def fast_neutron_flux2(solution_values, Ij, overlap_matrix):

    return solution_values[-1,0,0,0]#jnp.sum(solution_values[0,0,-1,:])

    return jnp.sum(last_element_solvals)
    
    egs_fast             = overlap_matrix @ last_element_solvals  #[1, n_basis_per_element]
    return jnp.sum(egs_fast)
    return jnp.sum(egs_fast)
    return jnp.einsum("ij, j->", egs_fast, Ij)  / jnp.sqrt(4 * jnp.pi) * 1e2

overlap_matrix = compute_overlap_matrix(groups_list , [14.2e6,1e6])
print(overlap_matrix)

@eqx.filter_jit
def objective(multigroup_objective, values_init, overlap_matrix):
    solution_values = multigroup_objective.solution(values_init).value
    return fast_neutron_flux2(solution_values, multigroup_objective.theta_parametrised_module.base_module.transport_operator.solution_domain_data.fem_matrices.integration_coeff, overlap_matrix)

grad_objective = eqx.filter_jit(jax.jacrev(objective, argnums=1))

objective(multigroup_objective, values_init, overlap_matrix)

grad_objective(multigroup_objective, values_init, overlap_matrix)

In [ ]:
%timeit grad_objective(multigroup_objective, values_init).bzshield.block_until_ready()

In [ ]:
@eqx.filter_jit
def objective(multigroup_objective, values_init):
    solution_values = multigroup_objective.solution(values_init).value
    return jnp.sum(solution_values), tritium_breeding_ratio(solution_values, multigroup_objective.theta_parametrised_module.additional_settings, multigroup_objective.theta_parametrised_module.base_module.transport_operator.transport_operator.solution_domain_data, multigroup_objective.aux_data), \
        fast_neutron_flux(solution_values, multigroup_objective.theta_parametrised_module.base_module.transport_operator.solution_domain_data.fem_matrices.integration_coeff, multigroup_objective.energy_set_boundaries)

grad_objective = eqx.filter_jit(jax.jacrev(objective, argnums=1))

objective(multigroup_objective, values_init)

#grad_objective(multigroup_objective, values_init)

In [ ]:
grad_objective(multigroup_objective, values_init)

In [ ]:
%timeit objective(multigroup_objective, values_init)[1].block_until_ready()

In [ ]:
%timeit grad_objective(multigroup_objective, values_init)[0].bzshield.block_until_ready()

In [ ]:
@eqx.filter_jit
def objective(multigroup_objective, values_init):
    return jnp.sum(multigroup_objective.solution(values_init).value)

grad_objective = eqx.filter_jit(jax.jacrev(objective, argnums=1))
grad_objective(multigroup_objective, values_init)

In [ ]:
objective(multigroup_objective, values_init)

In [ ]:
multigroup_objective.objective_and_constraints(values_init)

In [ ]:
from breeding_blanket_setup import _get_solution_interp

x, solution_values = _get_solution_interp(get_solution_domain(multigroup_objective.to_module(values_init), finite_element), multigroup_objective.solution(values_init).value)

In [ ]:
solution_values_collapsed  = (overlap_matrix @ solution_values)

In [ ]:
openmc_values_collapsed = overlap_matrix @  onp.array([tally_result_dict['flux']['mean'][...,i] for i in range(tally_result_dict['flux']['mean'].shape[-1])]) 

In [ ]:
fast_flux_constraint = 1e4 * 1e9 # neutrons/m2/s [1e9 cm2 / s]
                                 # attenuation factor 
flux_at_fw = 2* 10**18
attenuation_from_3d = fast_flux_constraint / flux_at_fw


In [ ]:
0.2  * 1e2 * onp.sqrt(4 * onp.pi)

In [ ]:
n_neutrons_stell = 3e9 / (1.6022e-19 * 14.1e6)
inner_wall_volume = jnp.pi * 1.0**2 * 2.0 * 22 * jnp.pi 
n_flux_plasma_avg = n_neutrons_stell / inner_wall_volume
print(n_flux_plasma_avg)

In [ ]:
for i in range(openmc_values_collapsed.shape[0]):    
    plt.stairs(openmc_values_collapsed[i], tally_result_dict['flux']['x'])
plt.yscale('log')
plt.plot((x- 0.5 * jnp.max(x)) * 100, solution_values_collapsed.T * 1e2 / onp.sqrt(4 * onp.pi))
plt.xlim(0)

In [ ]:
@eqx.filter_jit
def objective_function(base_obj : BZShieldMultiGroupObjective, theta : BZShieldValues):
    return base_obj.objective(theta)

dobj_dtheta = eqx.filter_jit(jax.grad(objective_function, argnums=1))

In [ ]:
multigroup_objective.objective(values_init)